# 01 — Dataset Overview

Pergunta: **o que cada fonte realmente contem e quais comparacoes sao validas?**

Schemas e labels originais sao preservados. FactChecks.br e uma colecao de tarefas heterogeneas; o FakeRecogna independente permanece separado.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, ensure_output_directories
from src.data.make_dataset import build_interim_datasets, load_interim_datasets

cfg = load_config()
ensure_output_directories(cfg)
if not list(Path(cfg["data"]["interim"]).glob("*.parquet")):
    build_interim_datasets()
datasets = {str(frame["dataset"].iloc[0]): frame for frame in load_interim_datasets().values()}
list(datasets)


In [ ]:
from src.data.load_data import load_factchecks, load_fakerecogna, discover_factchecks_files
from src.analysis.profiling import dataset_overview, label_distribution, top_values

raw = {**load_factchecks(include_aliases=True), "fakerecogna_original": load_fakerecogna()}
pd.DataFrame({"arquivo": [p.name for p in discover_factchecks_files()], "bytes": [p.stat().st_size for p in discover_factchecks_files()]})

## Schemas, tipos e amostras

Nao pressupomos colunas comuns e nao eliminamos campos antes da auditoria.

In [ ]:
for name, frame in raw.items():
    print(f"\n### {name}: {frame.shape[0]:,} linhas x {frame.shape[1]} colunas")
    display(frame.dtypes.rename("dtype").to_frame())
    display(frame.head(3))

## Variavel alvo e labels

`is_fake` e o alvo no release consolidado; `Classe` e o alvo no FakeRecogna original. A mesma codificacao numerica nao possui a mesma semantica entre fontes.

In [ ]:
pd.concat([label_distribution(frame, name) for name, frame in datasets.items()], ignore_index=True)

## Periodo temporal e principais fontes

In [ ]:
periods, sources = [], []
for name, frame in datasets.items():
    dates = frame["date_parsed"].dropna()
    periods.append({"dataset": name, "datas_validas": len(dates), "inicio": dates.min() if len(dates) else pd.NaT, "fim": dates.max() if len(dates) else pd.NaT})
    sources.append(top_values(frame, "domain", name, limit=10))
display(pd.DataFrame(periods))
display(pd.concat(sources, ignore_index=True))

## Resumo comparativo

Informacoes ausentes permanecem vazias; nada e inferido.

In [ ]:
dataset_overview(datasets).rename(columns={"dataset": "Dataset", "records": "Registros", "raw_labels": "Classes", "period_start": "Inicio", "period_end": "Fim", "main_columns": "Colunas principais"})